In [1]:
import os
import pandas as pd
from datetime import datetime
import json
import gc

folder_path_demanddetails = '/home/prerna/Punjab/punjab-data-prod-analysis/patiala/output_demand_details/'

# read active properties & needed columns
property_df = pd.read_csv(
    '/home/prerna/Punjab/punjab-data-prod-analysis/patiala/eg_pt_property.csv',
    usecols=['id', 'propertyid', 'tenantid', 'createdtime', 'additionaldetails', 'ownershipcategory', 'status', 'usagecategory']
)
property_df = property_df[property_df['status'] == 'ACTIVE'].copy()

# read units
# unit_df = pd.read_csv(
#     '/home/prerna/Punjab/punjab-data-prod-analysis/srihargobindpur/eg_pt_unit.csv',
#     usecols=['propertyid', 'occupancytype']
# )



# read demand
demand_df = pd.read_csv(
    '/home/prerna/Punjab/punjab-data-prod-analysis/patiala/egbs_demand_v1.csv',
    dtype={"consumercode": str},
    low_memory=False,
    usecols=['id', 'taxperiodfrom', 'taxperiodto', 'consumercode', 'status', 'businessservice']
)
demand_df = demand_df[demand_df['status'] == 'ACTIVE'].copy()
demand_df = demand_df[demand_df['businessservice'] == 'PT'].copy()


# read demand details (memory‑efficient, in chunks)
all_chunks = []
needed_cols = ['demandid', 'taxamount', 'collectionamount', 'taxheadcode']
for filename in os.listdir(folder_path_demanddetails):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path_demanddetails, filename)
        print(f'Loading: {file_path}')
        chunk = pd.read_csv(file_path, usecols=needed_cols)
        all_chunks.append(chunk)
demand_details_df = pd.concat(all_chunks, ignore_index=True)
del all_chunks; gc.collect()

print("✅ Loaded data")

Loading: /home/prerna/Punjab/punjab-data-prod-analysis/patiala/output_demand_details/output_263.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/patiala/output_demand_details/output_4.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/patiala/output_demand_details/output_34.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/patiala/output_demand_details/output_193.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/patiala/output_demand_details/output_79.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/patiala/output_demand_details/output_38.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/patiala/output_demand_details/output_292.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/patiala/output_demand_details/output_6.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/patiala/output_demand_details/output_201.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/patiala/output_demand_details/output_161.csv
Loading

In [2]:
print(len(property_df))         # number of rows in properties
# print(len(unit_df))             # number of rows in units
print(len(demand_df))   # number of rows in demand details
print(len(demand_details_df))   # number of rows in demand details

136121
916345
14643753


In [3]:
# join demand and demand details
joined_demand = demand_df.merge(demand_details_df, left_on='id', right_on='demandid', how='left', suffixes=('_demand', '_detail'))
print(joined_demand['id'].nunique())
del demand_details_df, demand_df; gc.collect()
joined_demand.head()

916345


,id,consumercode,businessservice,taxperiodfrom,taxperiodto,status,demandid,taxheadcode,taxamount,collectionamount
0,29608,PT-1909-016884,PT,1522540800000,1554076799000,ACTIVE,29608,PT_TAX,1196.67,1196.67
1,29608,PT-1909-016884,PT,1522540800000,1554076799000,ACTIVE,29608,PT_UNIT_USAGE_EXEMPTION,0.00,0.00
2,29608,PT-1909-016884,PT,1522540800000,1554076799000,ACTIVE,29608,PT_OWNER_EXEMPTION,0.00,0.00
3,29608,PT-1909-016884,PT,1522540800000,1554076799000,ACTIVE,29608,PT_FIRE_CESS,0.00,0.00
4,29608,PT-1909-016884,PT,1522540800000,1554076799000,ACTIVE,29608,PT_CANCER_CESS,23.94,23.94


In [4]:
import pytz

# Correct: parse as datetime from milliseconds since epoch
joined_demand['taxperiodfrom'] = pd.to_datetime(joined_demand['taxperiodfrom'], unit='ms', utc=True)
joined_demand['taxperiodto'] = pd.to_datetime(joined_demand['taxperiodto'], unit='ms', utc=True)

# Convert to IST (Asia/Kolkata)
ist = pytz.timezone('Asia/Kolkata')
joined_demand['taxperiodfrom'] = joined_demand['taxperiodfrom'].dt.tz_convert(ist)
joined_demand['taxperiodto'] = joined_demand['taxperiodto'].dt.tz_convert(ist)

# Financial year calculation
def get_fy(date):
    if date.month >= 4:
        fy_start = date.year
        fy_end = date.year + 1
    else:
        fy_start = date.year - 1
        fy_end = date.year
    return f"{fy_start}-{str(fy_end)[-2:]}"

joined_demand['fy'] = joined_demand['taxperiodfrom'].apply(get_fy)

# Group by consumercode
result = joined_demand.groupby('consumercode')['fy'].agg(['min', 'max']).reset_index()
result.rename(columns={'min': 'earliest_fy', 'max': 'latest_fy'}, inplace=True)

print(result)

           consumercode earliest_fy latest_fy
0        PT-1909-015023     2018-19   2024-25
1        PT-1909-016884     2018-19   2024-25
2        PT-1909-031318     2017-18   2020-21
3        PT-1909-035743     2018-19   2018-19
4        PT-1909-046124     2018-19   2024-25
...                 ...         ...       ...
135860  PT-1910-2466161     2013-14   2025-26
135861  PT-1910-2466215     2022-23   2025-26
135862  PT-1910-2466253     2021-22   2025-26
135863  PT-1910-2466389     2018-19   2025-26
135864  PT-1910-2466445     2022-23   2025-26

[135865 rows x 3 columns]


In [5]:
# Merge latest_fy onto joined_demand by consumercode
joined = joined_demand.merge(
    result[['consumercode', 'latest_fy']],
    on='consumercode',
    how='left'
)

# Filter only latest FY
latest_demand = joined[joined['fy'] == joined['latest_fy']]

# Pivot taxheadcode values into separate columns
pivoted = latest_demand.pivot_table(
    index='consumercode',
    columns='taxheadcode',
    values='taxamount',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Apply formula:
# PT_TAX + PT_CANCER_CESS + PT_FIRE_CESS + PT_ROUNDOFF - (PT_OWNER_EXEMPTION + PT_UNIT_USAGE_EXEMPTION)
pivoted['latest_fy_taxamount'] = (
    pivoted.get('PT_TAX', 0) +
    pivoted.get('PT_CANCER_CESS', 0) +
    pivoted.get('PT_FIRE_CESS', 0) +
    pivoted.get('PT_ROUNDOFF', 0) -
    ( pivoted.get('PT_OWNER_EXEMPTION', 0).abs() + pivoted.get('PT_UNIT_USAGE_EXEMPTION', 0).abs() )
)

# Merge back into result
result = result.merge(
    pivoted[['consumercode', 'latest_fy_taxamount']],
    on='consumercode',
    how='left'
)

print(result.head())


     consumercode earliest_fy latest_fy  latest_fy_taxamount
0  PT-1909-015023     2018-19   2024-25                 0.00
1  PT-1909-016884     2018-19   2024-25              1483.23
2  PT-1909-031318     2017-18   2020-21                 0.00
3  PT-1909-035743     2018-19   2018-19               362.44
4  PT-1909-046124     2018-19   2024-25              1116.55


In [6]:
# Calculating the tax amount (demand) of current year using formula
target_fy = "2025-26"
current_fy_demand = joined_demand[joined_demand['fy'] == target_fy]

# Pivot taxheadcode values into separate columns
pivoted_current = current_fy_demand.pivot_table(
    index='consumercode',
    columns='taxheadcode',
    values='taxamount',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Apply formula:
pivoted_current['current_fy_taxamount'] = (
    pivoted_current.get('PT_TAX', 0) +
    pivoted_current.get('PT_CANCER_CESS', 0) +
    pivoted_current.get('PT_FIRE_CESS', 0) +
    pivoted_current.get('PT_ROUNDOFF', 0) -
    ( pivoted_current.get('PT_OWNER_EXEMPTION', 0).abs() + pivoted_current.get('PT_UNIT_USAGE_EXEMPTION', 0).abs() )
)

# Keep only required cols
pivoted_current = pivoted_current[['consumercode', 'current_fy_taxamount']]

# Ensure all consumercodes are present
all_consumercodes = pd.DataFrame(joined_demand['consumercode'].unique(), columns=['consumercode'])
final = all_consumercodes.merge(pivoted_current, on='consumercode', how='left')
final['current_fy_taxamount'] = final['current_fy_taxamount'].fillna(0)

# Merge into result
result = result.merge(final, on='consumercode', how='left')
result['current_fy_taxamount'] = result['current_fy_taxamount'].fillna(0)

print(result.head())


     consumercode earliest_fy latest_fy  latest_fy_taxamount  \
0  PT-1909-015023     2018-19   2024-25                 0.00   
1  PT-1909-016884     2018-19   2024-25              1483.23   
2  PT-1909-031318     2017-18   2020-21                 0.00   
3  PT-1909-035743     2018-19   2018-19               362.44   
4  PT-1909-046124     2018-19   2024-25              1116.55   

   current_fy_taxamount  
0                   0.0  
1                   0.0  
2                   0.0  
3                   0.0  
4                   0.0  


In [7]:
property_result_merged = property_df.merge(
    result,
    left_on='propertyid',
    right_on='consumercode',
    how='left'
)

print(property_result_merged)

                                          id       propertyid    tenantid  \
0       7b7593ba-d9fd-4ade-9b62-4c84e59a7885  PT-1910-1380786  pb.patiala   
1       38e60ced-0142-44ef-9162-839e3844c79a  PT-1910-1171983  pb.patiala   
2       edc59d90-4578-4fe5-8988-c8eb90ea9601  PT-1910-1170452  pb.patiala   
3       fa06ba77-2491-4433-bd10-67c24bd0a768  PT-1910-1178924  pb.patiala   
4       325d1f7d-21ca-494a-bc8e-40b4480b58ee  PT-1910-1196167  pb.patiala   
...                                      ...              ...         ...   
136116  1e2da04b-d2d4-484d-991c-4591fe07a4c8  PT-1910-1395438  pb.patiala   
136117  f33cb80b-ed0a-4744-b937-5eb8edfad678  PT-1910-1343905  pb.patiala   
136118  9e634662-656a-4a77-9811-5ef799b4964c  PT-1910-1294308  pb.patiala   
136119  2e75c8c4-6649-495c-9d54-87d7beaba651  PT-1910-1240383  pb.patiala   
136120  7e4899f4-a465-4c70-a8b6-ced63c8f9108  PT-1910-1289747  pb.patiala   

        status          ownershipcategory              usagecategory  \
0  

In [8]:
property_result_merged.to_csv('Punjab_Data_Analysis_patiala_final_2.csv', index=False)